# Step25 — BGE-M3 frozen anchor extraction

Self-contained notebook for Modal/Colab-style notebook runtimes. It extracts frozen `BAAI/bge-m3` features for train/public/private and writes `outputs/bge_m3_frozen_anchor/` for the local Step25 stack script.

## 0. Runtime notes

Upload either the full project folder or `asp_data_v3.zip` / `asp_data.zip` into the notebook environment before running. If your zip extracts into a nested folder, the auto-detect cell will find the folder containing `data/raw/train.csv`.

In [ ]:
# Install dependencies. Re-run kernel after install only if the notebook runtime asks for it.
import sys, subprocess, importlib.util

packages = ['FlagEmbedding', 'transformers', 'accelerate', 'sentencepiece', 'einops', 'tqdm']
missing = [p for p in packages if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', *missing])
else:
    print('All dependencies already installed.')

In [ ]:
from __future__ import annotations

import json
import os
import re
import shutil
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import torch
    print('torch:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('torch check failed:', exc)

# Optional: set ZIP_PATH manually if auto-detect does not find your upload.
ZIP_PATH = None  # e.g. '/root/asp_data_v3.zip' or '/content/drive/MyDrive/asp_data_v3.zip'
WORK_DIR = Path('/tmp/asp_step25_work')
MODEL_NAME = 'BAAI/bge-m3'
USE_FP16 = True
BATCH_SIZE = 8
MAX_LENGTH = 8192

if 'COLAB_GPU' in os.environ:
    BATCH_SIZE = 4
    MAX_LENGTH = 4096

print({'BATCH_SIZE': BATCH_SIZE, 'MAX_LENGTH': MAX_LENGTH, 'USE_FP16': USE_FP16})

In [ ]:
def has_raw_layout(path: Path) -> bool:
    return (path / 'data' / 'raw' / 'train.csv').exists()

def has_flat_layout(path: Path) -> bool:
    required = ['train.csv', 'public_test.csv', 'private_test.csv', 'Test_Submission.csv']
    return all((path / name).exists() for name in required)

def find_project_root(start_dirs: list[Path]) -> Path | None:
    candidates = []
    for start in start_dirs:
        if not start.exists():
            continue
        for path in [start, *start.rglob('*')]:
            if path.is_dir() and (has_raw_layout(path) or has_flat_layout(path)):
                candidates.append(path)
    if not candidates:
        return None
    return sorted(candidates, key=lambda path: (0 if has_raw_layout(path) else 1, len(str(path))))[0]

def find_zip() -> Path | None:
    if ZIP_PATH is not None:
        p = Path(ZIP_PATH)
        if not p.exists():
            raise FileNotFoundError(p)
        return p
    search_roots = [Path.cwd(), Path('/root'), Path('/content'), Path('/mnt/data'), Path('/tmp')]
    zips = []
    for root in search_roots:
        if root.exists():
            zips.extend(root.glob('asp_data_v3.zip'))
            zips.extend(root.glob('asp_data.zip'))
            zips.extend(root.rglob('asp_data_v3.zip'))
            zips.extend(root.rglob('asp_data.zip'))
    if not zips:
        return None
    zips = sorted(set(zips), key=lambda p: (0 if 'v3' in p.name else 1, len(str(p))))
    return zips[0]

ROOT = find_project_root([Path.cwd(), Path('/root'), Path('/content'), Path('/mnt/data'), Path('/tmp')])
if ROOT is None:
    zip_path = find_zip()
    if zip_path is None:
        raise FileNotFoundError('Could not find project root or asp_data_v3.zip/asp_data.zip. Upload the zip, or set ZIP_PATH manually.')
    print('Extracting', zip_path, 'to', WORK_DIR)
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(WORK_DIR)
    ROOT = find_project_root([WORK_DIR])

if ROOT is None:
    raise FileNotFoundError('Zip extracted, but no folder containing either data/raw/train.csv or flat train.csv/public_test.csv files was found.')

if has_raw_layout(ROOT):
    DATA_DIR = ROOT / 'data' / 'raw'
    EXTERNAL_DIR = ROOT / 'outputs' / 'external'
    OUTPUT_ROOT = ROOT / 'outputs'
else:
    DATA_DIR = ROOT
    EXTERNAL_DIR = ROOT
    OUTPUT_ROOT = ROOT / 'outputs'
OUT = OUTPUT_ROOT / 'bge_m3_frozen_anchor'
OUT.mkdir(parents=True, exist_ok=True)
SRC_DIR = ROOT / 'src'
if SRC_DIR.exists():
    sys.path.append(str(SRC_DIR))

print('ROOT =', ROOT)
print('DATA_DIR =', DATA_DIR)
print('OUT =', OUT)
print('raw files:', [p.name for p in DATA_DIR.glob('*.csv')])

In [ ]:
try:
    from step18b_targeted_feature_calibration import attach_abstracts, load_abstracts
    print('Using attach_abstracts/load_abstracts from src.')
except Exception as exc:
    print('Using fallback abstract helpers because src import failed:', exc)

    def load_abstracts() -> pd.DataFrame:
        for name in ['abstracts_merged_v3.csv', 'abstracts_merged_v2.csv', 'abstracts_merged.csv']:
            path = EXTERNAL_DIR / name
            if path.exists():
                print('abstract file:', path)
                return pd.read_csv(path)
        print('No abstracts_merged*.csv found; using empty abstracts.')
        return pd.DataFrame(columns=['source_split', 'id', 'abstract', 'has_abstract', 's2_venue', 's2_publication_types', 's2_fields_of_study'])

    def attach_abstracts(df: pd.DataFrame, split: str, abstracts: pd.DataFrame) -> pd.DataFrame:
        if len(abstracts) and 'source_split' in abstracts.columns:
            cols = [c for c in ['id', 'abstract', 'has_abstract', 'abstract_len', 's2_venue', 's2_publication_types', 's2_fields_of_study', 'oa_type', 'cr_type'] if c in abstracts.columns]
            sub = abstracts[abstracts['source_split'].eq(split)][cols].copy()
            df = df.merge(sub, on='id', how='left')
        for col, default in {
            'abstract': '', 'has_abstract': False, 'abstract_len': 0, 's2_venue': '',
            's2_publication_types': '', 's2_fields_of_study': '', 'oa_type': '', 'cr_type': '',
        }.items():
            if col not in df.columns:
                df[col] = default
            df[col] = df[col].fillna(default)
        df['has_abstract'] = df['has_abstract'].astype(bool)
        return df

In [ ]:
def load_split(split: str) -> pd.DataFrame:
    file_name = {'train': 'train.csv', 'public': 'public_test.csv', 'private': 'private_test.csv'}[split]
    abstract_split = {'train': 'train', 'public': 'public_test', 'private': 'private_test'}[split]
    df = pd.read_csv(DATA_DIR / file_name)
    return attach_abstracts(df, abstract_split, load_abstracts())

def clean(value) -> str:
    if pd.isna(value):
        return ''
    return str(value).strip()

def build_text(row: pd.Series) -> str:
    parts = [
        f"Title: {clean(row.get('title', ''))}",
        f"Venue: {clean(row.get('venue', ''))}",
        f"Authors: {clean(row.get('authors', ''))}",
    ]
    abstract = clean(row.get('abstract', ''))
    if abstract:
        parts.append(f"Abstract: {abstract}")
    for label, col in [('Semantic Scholar venue', 's2_venue'), ('Publication types', 's2_publication_types'), ('Fields of study', 's2_fields_of_study')]:
        value = clean(row.get(col, ''))
        if value:
            parts.append(f"{label}: {value}")
    return '\n'.join(parts)

train = load_split('train')
public = load_split('public')
private = load_split('private')

for name, df in [('train', train), ('public', public), ('private', private)]:
    if df['id'].duplicated().any():
        raise ValueError(f'{name} has duplicate ids')
    df['bge_m3_text'] = df.apply(build_text, axis=1)
    print(name, 'rows=', len(df), 'has_abstract=', int(df['has_abstract'].sum()), 'avg_chars=', round(df['bge_m3_text'].str.len().mean(), 1))

pd.DataFrame({'id': train['id'], 'text': train['bge_m3_text']}).to_csv(OUT / 'train_texts.csv', index=False)
pd.DataFrame({'id': public['id'], 'text': public['bge_m3_text']}).to_csv(OUT / 'public_texts.csv', index=False)
pd.DataFrame({'id': private['id'], 'text': private['bge_m3_text']}).to_csv(OUT / 'private_texts.csv', index=False)

In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel(MODEL_NAME, use_fp16=USE_FP16)

def encode_texts(texts: list[str], name: str):
    print(f'encoding {name}: {len(texts)} rows')
    output = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
    )
    dense = np.asarray(output['dense_vecs'], dtype=np.float32)
    sparse = output['lexical_weights']
    np.save(OUT / f'{name}_dense.npy', dense)
    print(name, 'dense shape =', dense.shape)
    return dense, sparse

train_dense, train_sparse = encode_texts(train['bge_m3_text'].tolist(), 'train')
public_dense, public_sparse = encode_texts(public['bge_m3_text'].tolist(), 'public')
private_dense, private_sparse = encode_texts(private['bge_m3_text'].tolist(), 'private')

train[['id']].to_csv(OUT / 'train_ids.csv', index=False)
public[['id']].to_csv(OUT / 'public_ids.csv', index=False)
private[['id']].to_csv(OUT / 'private_ids.csv', index=False)

In [ ]:
RUBRICS = {
    'class1_irrelevant': 'Papers not about Answer Set Programming, Logic Programming, Knowledge Representation, nonmonotonic reasoning, stable model semantics, Datalog, ASP solvers, or closely related KR and LP methods.',
    'class2_adjacent': 'Papers adjacent to logic, formal methods, planning, verification, theorem proving, databases, ontologies, or AI reasoning, but not centrally about ASP, LP, KR, stable models, or nonmonotonic reasoning.',
    'class3_related': 'Papers related to Knowledge Representation, Logic Programming, argumentation, abduction, Datalog, planning, ontologies, or symbolic AI methods with some connection to ASP or nonmonotonic reasoning.',
    'class4_relevant': 'Papers clearly relevant to Answer Set Programming, Logic Programming, stable model semantics, nonmonotonic reasoning, ASP solving, grounding, KR formalisms, or direct ASP applications.',
    'class5_core': 'Core papers on Answer Set Programming, answer sets, stable model semantics, ASP solvers such as clingo or DLV, grounding, disjunctive logic programs, nonmonotonic logic programming, and central theoretical or solver advances.',
}

rubric_output = model.encode(
    list(RUBRICS.values()),
    batch_size=5,
    max_length=MAX_LENGTH,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False,
)
rubric_dense = np.asarray(rubric_output['dense_vecs'], dtype=np.float32)
rubric_sparse = rubric_output['lexical_weights']
np.save(OUT / 'rubric_dense.npy', rubric_dense)
pd.DataFrame({'rubric': list(RUBRICS.keys()), 'text': list(RUBRICS.values())}).to_csv(OUT / 'rubrics.csv', index=False)

def dense_scores(dense: np.ndarray) -> pd.DataFrame:
    scores = dense @ rubric_dense.T
    return pd.DataFrame(scores, columns=[f'bge_m3_dense_{k}' for k in RUBRICS])

def sparse_scores(sparse) -> pd.DataFrame:
    rows = []
    for item in tqdm(sparse, desc='sparse rubric scoring'):
        rows.append([model.compute_lexical_matching_score(item, r) for r in rubric_sparse])
    return pd.DataFrame(rows, columns=[f'bge_m3_sparse_{k}' for k in RUBRICS])

for name, df, dense, sparse in [
    ('train', train, train_dense, train_sparse),
    ('public', public, public_dense, public_sparse),
    ('private', private, private_dense, private_sparse),
]:
    feat = pd.concat([df[['id']].reset_index(drop=True), dense_scores(dense), sparse_scores(sparse)], axis=1)
    feat.to_csv(OUT / f'{name}_diagnostics.csv', index=False)
    print(name, 'diagnostics shape =', feat.shape)

In [ ]:
required = [
    'train_dense.npy', 'public_dense.npy', 'private_dense.npy',
    'train_ids.csv', 'public_ids.csv', 'private_ids.csv',
    'train_diagnostics.csv', 'public_diagnostics.csv', 'private_diagnostics.csv',
]
missing = [name for name in required if not (OUT / name).exists()]
if missing:
    raise FileNotFoundError(missing)

manifest = {
    'method': 'bge_m3_frozen_anchor',
    'model': MODEL_NAME,
    'root': str(ROOT),
    'batch_size': BATCH_SIZE,
    'max_length': MAX_LENGTH,
    'use_fp16': USE_FP16,
    'train_rows': int(len(train)),
    'public_rows': int(len(public)),
    'private_rows': int(len(private)),
    'train_has_abstract': int(train['has_abstract'].sum()),
    'public_has_abstract': int(public['has_abstract'].sum()),
    'private_has_abstract': int(private['has_abstract'].sum()),
    'train_dense_shape': list(train_dense.shape),
    'public_dense_shape': list(public_dense.shape),
    'private_dense_shape': list(private_dense.shape),
    'rubrics': list(RUBRICS.keys()),
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))

In [ ]:
archive_base = ROOT / 'outputs' / 'bge_m3_frozen_anchor_outputs'
archive_path = shutil.make_archive(str(archive_base), 'zip', OUT)
print('Created archive:', archive_path)
print('Download/copy this zip back to your local repo, then extract it into outputs/bge_m3_frozen_anchor/.')